# 911 Automate — End-to-End RAG Pipeline POC

This notebook demonstrates the full **Phase 1–6** pipeline end-to-end on a local machine using a sample PDF and an offline `MockLLM`.

| Phase | Component | Role |
|-------|-----------|------|
| 1 | **DocumentLoader** (`PyMuPDFDocumentLoader`) | PDF → LangChain `Document` objects |
| 2 | **Chunker** (`create_chunks`) | Documents → overlapping text chunks |
| 3 | **Embedder** (`all-MiniLM-L6-v2`) | Chunks → 384-dim vectors |
| 4 | **Qdrant upsert** (in-memory) | Vectors → local vector store |
| 5 | **Retriever** (cosine + MMR) | Query → top-k relevant chunks |
| 6 | **Agent + MockLLM** | Chunks → `answer` / `clarify` / `escalate` |

**Mock vs. Real LLM:** The notebook uses a `MockLLM` with canned responses.  See the last cell for a one-line swap to `OllamaLLM` or `APILLM`.

> **Requirements (one-time):** `pip install -r requirements.txt fpdf2`  > The embedder downloads `all-MiniLM-L6-v2` (~80 MB) on first run.

In [ ]:
import sys
import os
import json
import tempfile
from pathlib import Path

# ── project root → src.rag.* imports ──────────────────────────────────────
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root))

from src.rag.config import Config
from src.rag.ingestion import PyMuPDFDocumentLoader
from src.rag.chunking import create_chunks
from src.rag.embedder import Embedder
from src.rag.vectordb_qdrant import VectorDBQdrant
from src.rag.retriever import Retriever
from src.rag.agent import Agent
from src.rag.prompt_handler import PromptHandler
from qdrant_client import QdrantClient

# Temp directory for all artifacts produced by this notebook run
tmp_dir = Path(tempfile.mkdtemp(prefix="poc_911_"))
print(f"Temp dir : {tmp_dir}")

# Config with prompts_dir pointing at the real prompts folder
config = Config(
    collection_name="poc_demo",
    prompts_dir=str(project_root / "prompts"),
)
print(f"Config   : {config}")


In [ ]:
# ── Phase 1: Create a minimal EMS-protocol PDF with fpdf2 ─────────────────
from fpdf import FPDF

PAGE1_LINES = [
    "EMERGENCY CHILDBIRTH PROTOCOL",
    "",
    "Overview",
    "This protocol guides 911 dispatchers and first responders through",
    "out-of-hospital emergency delivery situations. Activate when a caller",
    "reports imminent birth.",
    "",
    "Step 1 - Assess Imminence",
    "Ask the caller: Does the mother feel an urge to push? Is the baby",
    "crowning (head visible)? If yes to either, delivery is imminent.",
    "",
    "Step 2 - Prepare the Environment",
    "Keep the mother calm. Have her lie on a flat surface. Gather clean",
    "towels or blankets to warm the newborn after birth.",
    "",
    "Step 3 - Assist the Delivery",
    "Support the baby's head gently as it emerges - do not pull. Guide",
    "downward for the first shoulder, then upward for the second.",
    "",
    "Step 4 - Newborn Airway and Breathing",
    "Place the baby skin-to-skin on the mother's chest. Dry and stimulate",
    "vigorously with a towel. The baby should cry within 30 seconds.",
    "If not, begin infant CPR and dispatch ALS immediately.",
]

PAGE2_LINES = [
    "POST-DELIVERY CARE AND ESCALATION",
    "",
    "Cord Management",
    "Do not cut the umbilical cord unless trained and necessary. If cutting",
    "is required, tie firmly at least 6 inches from the baby first.",
    "",
    "Placenta Delivery",
    "The placenta typically delivers 5-30 minutes after birth. Do not pull",
    "on the cord. Preserve the placenta for hospital evaluation.",
    "",
    "Maternal Hemorrhage",
    "If the mother has heavy bleeding (soaking more than one pad per hour),",
    "gently massage the lower abdomen. Keep the mother warm and monitor",
    "vital signs continuously.",
    "",
    "Escalation Criteria - Call for ALS Immediately",
    "Escalate to Advanced Life Support or hospital immediately when:",
    "  - Baby is not breathing after 30 seconds of stimulation",
    "  - Cord is wrapped around the baby's neck (nuchal cord)",
    "  - Breech presentation: feet or buttocks appear first",
    "  - Mother is unconscious or unresponsive",
    "  - Heavy maternal hemorrhage not slowing after uterine massage",
    "",
    "Documentation",
    "Record: time of delivery, Apgar score at 1 and 5 minutes, any",
    "complications, and all interventions. Relay to receiving hospital.",
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_margins(20, 20, 20)

for page_lines in [PAGE1_LINES, PAGE2_LINES]:
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)
    for line in page_lines:
        if line == "":
            pdf.ln(4)
        elif line == line.upper() and len(line) > 5:   # section header heuristic
            pdf.set_font("Helvetica", "B", 12)
            pdf.multi_cell(0, 7, line)
            pdf.set_font("Helvetica", size=11)
        else:
            pdf.multi_cell(0, 6, line)

pdf_path = tmp_dir / "sample_ems.pdf"
pdf.output(str(pdf_path))
print(f"Sample PDF created : {pdf_path}")
print(f"File size          : {pdf_path.stat().st_size:,} bytes  (2 pages)")


In [ ]:
# ── Phase 2: PDF → LangChain Documents ────────────────────────────────────
print("Loading PDF via PyMuPDFDocumentLoader...")
loader = PyMuPDFDocumentLoader()
docs = loader.load_documents(
    pdf_path,
    doc_id="sample_ems",
    output_dir=tmp_dir / "markdown",
)

print(f"\nLoaded {len(docs)} page document(s)")
for doc in docs:
    print(f"\n--- Page {doc.metadata['page']} ---")
    print(f"  metadata : {doc.metadata}")
    print(f"  preview  : {doc.page_content[:220]!r}")


In [ ]:
# ── Phase 3: Documents → overlapping text chunks ──────────────────────────
print("Chunking documents...")
chunks = create_chunks(
    docs,
    chunk_size=400,
    chunk_overlap=80,
    output_dir=tmp_dir / "chunks",
)

print(f"\nProduced {len(chunks)} chunk(s)  "
      f"(chunk_size=400, chunk_overlap=80)")

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(f"  metadata : {chunk.metadata}")
    print(f"  length   : {len(chunk.page_content)} chars")
    print(f"  preview  : {chunk.page_content[:160]!r}")

if len(chunks) > 3:
    print(f"\n... and {len(chunks) - 3} more chunk(s)")


In [ ]:
# ── Phase 4: Chunks → 384-dim vectors (all-MiniLM-L6-v2) ─────────────────
# NOTE: Downloads the model (~80 MB) on the very first run; cached thereafter.
print("Initialising embedder (may download model on first run)...")
embedder = Embedder(config)

texts = [c.page_content for c in chunks]
print(f"Embedding {len(texts)} chunk(s)...")
vectors = embedder.embed_texts(texts)

print(f"\nEmbedding complete")
print(f"  Vector count  : {len(vectors)}")
print(f"  Dimensions    : {len(vectors[0]) if vectors else 0}")
first8 = [round(v, 4) for v in vectors[0][:8]] if vectors else []
print(f"  First 8 dims  : {first8}  ...")


In [ ]:
# ── Phase 5: Upsert into in-memory Qdrant (no server required) ────────────
print("Setting up in-memory Qdrant...")
vdb = VectorDBQdrant(config)
vdb._client = QdrantClient(":memory:")   # swap real client for in-memory

chunk_items = [
    {
        "id": i,
        "vector": vectors[i],
        "metadata": {
            "text": c.page_content,          # required by Retriever._resolve_text
            "doc_id": c.metadata.get("doc_id", ""),
            "page": c.metadata.get("page", 0),
            "chunk_index": c.metadata.get("chunk_index", i),
            "source": c.metadata.get("source", ""),
        },
    }
    for i, c in enumerate(chunks)
]

vdb.upsert_chunks(chunk_items)

info = vdb._client.get_collection(config.collection_name)
print(f"\nCollection   : {config.collection_name}")
print(f"  Points       : {info.points_count}")
print(f"  Vector size  : {info.config.params.vectors.size}")
print(f"  Distance     : {info.config.params.vectors.distance}")


In [ ]:
# ── Phase 6: Retriever — query → top-k chunks with scores ────────────────
retriever = Retriever(embedder, vdb, config)

query_high = "What are the steps to assist with emergency childbirth delivery?"
print(f"Query: {query_high!r}\n")

results = retriever.retrieve(query_high)

print(f"Top-{len(results)} retrieved chunks (cosine + MMR):")
print(f"{'Rank':>4}  {'Score':>6}  {'Page':>4}  {'Chunk':>5}  Preview")
print("-" * 70)
for i, r in enumerate(results):
    preview = r["text"][:60].replace("\n", " ")
    print(
        f"{i+1:>4}  {r['score']:>6.4f}  "
        f"{r['metadata'].get('page', '?'):>4}  "
        f"{r['metadata'].get('chunk_index', '?'):>5}  "
        f"{preview}..."
    )


## Mock LLM

`MockLLM` implements the same `invoke(messages) -> str` interface as `OllamaLLM` / `APILLM` and routes **canned responses** based on whether the system prompt is the *clarify* prompt or the *agent-system* prompt:

| Trigger | Response |
|---------|----------|
| `"clarif"` in system prompt | Two structured clarifying questions |
| Agent-system prompt | JSON with `"answer"` + `"confidence": 0.87` |

This lets the entire pipeline run **offline and deterministically** — ideal for testing the answer/clarify/escalate decision logic without a real LLM.

In [ ]:
class MockLLM:
    """
    Deterministic stand-in for OllamaLLM / APILLM.

    Routes canned responses by inspecting the system-prompt content.
    Implements the same invoke(messages) -> str interface, so it can be
    passed directly to Agent(..., llm=MockLLM()).
    """

    ANSWER_RESPONSE = json.dumps(
        {
            "answer": (
                "Keep the mother calm and have her lie on a flat surface. "
                "Support the baby's head gently as it emerges - do not pull. "
                "Place the newborn skin-to-skin on the mother's chest, dry it "
                "vigorously, and ensure it cries within 30 seconds. "
                "If the baby does not breathe, begin infant CPR and dispatch ALS."
            ),
            "confidence": 0.87,
        },
        indent=2,
    )

    CLARIFY_RESPONSE = (
        "Could you clarify a few things so I can assist more precisely?\n"
        "1. What is the patient's current condition — is the baby crowning "
        "or has delivery already begun?\n"
        "2. Are emergency paramedics already en route to the scene?"
    )

    def invoke(self, messages: list) -> str:
        """Return canned response: clarify questions or JSON answer."""
        system_content = next(
            (m["content"] for m in messages if m["role"] == "system"), ""
        )
        if "clarif" in system_content.lower():
            return self.CLARIFY_RESPONSE
        return self.ANSWER_RESPONSE


mock_llm = MockLLM()
prompt_handler = PromptHandler(config.prompts_dir)
agent = Agent(retriever, config, llm=mock_llm, prompt_handler=prompt_handler)

print("MockLLM ready.")
print(f"\nAnswer response preview  : {MockLLM.ANSWER_RESPONSE[:90]}...")
print(f"\nClarify response preview : {MockLLM.CLARIFY_RESPONSE[:90]}...")


In [ ]:
# ── Flow 1: ANSWER ────────────────────────────────────────────────────────
# High-confidence query → retrieval score above threshold → agent answers
print("=" * 62)
print("FLOW 1: ANSWER  (retrieval confidence >= threshold)")
print("=" * 62)

query_answer = "What are the steps to assist with emergency childbirth delivery?"
print(f"\nQuery: {query_answer!r}")

# Inspect retrieval scores before the agent runs
chunks_ans = retriever.retrieve(query_answer)
scores_ans = [r["score"] for r in chunks_ans]
mean_ans = sum(scores_ans) / len(scores_ans) if scores_ans else 0.0
print(f"\nRetrieval scores : {[round(s, 4) for s in scores_ans]}")
print(f"Mean confidence  : {mean_ans:.4f}  (threshold={config.confidence_threshold})")

result_ans = agent.handle(query_answer, session={})

print(f"\n--- Agent Decision ---")
print(f"  action     : {result_ans['action']!r}")
print(f"  confidence : {result_ans['confidence']:.4f}")
print(f"  response   :")
print(result_ans["response"])


In [ ]:
# ── Flow 2: CLARIFY ───────────────────────────────────────────────────────
# Vague/off-topic query → low retrieval confidence → agent asks for more info
# clarify_rounds=0 → should_escalate() is False → action="clarify"
print("=" * 62)
print("FLOW 2: CLARIFY  (low confidence, clarify_rounds < max)")
print("=" * 62)

query_vague = "I need some help, something is happening"
print(f"\nQuery: {query_vague!r}")

chunks_vag = retriever.retrieve(query_vague)
scores_vag = [r["score"] for r in chunks_vag]
mean_vag = sum(scores_vag) / len(scores_vag) if scores_vag else 0.0
print(f"\nRetrieval scores : {[round(s, 4) for s in scores_vag]}")
print(f"Mean confidence  : {mean_vag:.4f}  (threshold={config.confidence_threshold})")

session_c = {"clarify_rounds": 0}
result_cla = agent.handle(query_vague, session=session_c)

print(f"\n--- Agent Decision ---")
print(f"  action         : {result_cla['action']!r}")
print(f"  confidence     : {result_cla['confidence']:.4f}")
print(f"  clarify_rounds : {result_cla['session']['clarify_rounds']} "
      f"(was 0, now incremented)")
print(f"  response       :")
print(result_cla["response"])


In [ ]:
# ── Flow 3: ESCALATE ──────────────────────────────────────────────────────
# Same low-confidence query but clarify_rounds already exhausted
# should_escalate(confidence < threshold, rounds >= max) → True → escalate
print("=" * 62)
print("FLOW 3: ESCALATE  (clarify rounds exhausted)")
print("=" * 62)

print(f"\nQuery: {query_vague!r}")
print(f"  injected clarify_rounds : {config.max_clarify_rounds}  "
      f"(== max_clarify_rounds)")
print(f"  confidence              : {mean_vag:.4f} < {config.confidence_threshold}")

# Carry forward the conversation history from the clarify round above
session_e = {
    "clarify_rounds": config.max_clarify_rounds,
    "history": result_cla["session"]["history"],
}
result_esc = agent.handle(query_vague, session=session_e)

print(f"\n--- Agent Decision ---")
print(f"  action     : {result_esc['action']!r}")
print(f"  confidence : {result_esc['confidence']:.4f}")
print(f"  response   :")
print(result_esc["response"])

# Summarise all three flows
print("\n" + "=" * 62)
print("FLOW SUMMARY")
print("=" * 62)
print(f"  answer   → action={result_ans['action']!r},  "
      f"confidence={result_ans['confidence']:.4f}")
print(f"  clarify  → action={result_cla['action']!r}, "
      f"confidence={result_cla['confidence']:.4f}, "
      f"rounds={result_cla['session']['clarify_rounds']}")
print(f"  escalate → action={result_esc['action']!r}")


In [ ]:
# ── Swapping MockLLM for a Real LLM ──────────────────────────────────────
#
# The only change needed is the `llm=` argument to Agent().
# Everything else (retriever, config, prompt_handler) stays the same.
#
# Option A: Local Ollama (requires `ollama pull llama3.2`)
# --------------------------------------------------------
# from src.rag.llm_adapters import OllamaLLM
#
# real_agent = Agent(
#     retriever,
#     config,
#     llm=OllamaLLM(config),
#     prompt_handler=prompt_handler,
# )
#
# Option B: OpenAI-compatible API (OpenAI, Together.ai, Groq, etc.)
# -----------------------------------------------------------------
# from src.rag.llm_adapters import APILLM
#
# api_config = Config(
#     collection_name="poc_demo",
#     prompts_dir=str(project_root / "prompts"),
#     api_base_url="https://api.openai.com/v1",
#     api_key="sk-...",
#     api_model="gpt-4o-mini",
# )
# real_agent = Agent(
#     retriever,
#     api_config,
#     llm=APILLM(api_config),
#     prompt_handler=prompt_handler,
# )
#
# Then call identically:
# result = real_agent.handle(
#     "What are the steps for emergency childbirth?", session={}
# )
# print(result["action"], result["response"])

print("Pipeline run complete.  Swap MockLLM for a real LLM using the")
print("commented-out blocks above (Option A: Ollama, Option B: API).")
print()
print("Pipeline artefact summary:")
print(f"  Documents loaded  : {len(docs)}")
print(f"  Chunks created    : {len(chunks)}")
print(f"  Vectors stored    : {len(vectors)}")
print(f"  Retriever top-k   : {config.top_k}")
print(f"  Confidence thr.   : {config.confidence_threshold}")
print(f"  Max clarify rds   : {config.max_clarify_rounds}")
print(f"  Temp artefacts    : {tmp_dir}")
